# Build the feature cache: ds004504, all 88 subjects

This notebook turns the raw EEG in `data/` into `subject_cache/`. Every model
notebook (`GAT_ds004504`, `GCN_ds004504`, `RF_ds004504`, the multi-class GAT)
reads only that cache.

It is the data half of `mfe-fcgcn.ipynb`. The PSD, normalisation, connectivity
and window-building cells are **copied verbatim**, so new subjects are processed
exactly like the existing ones. Cell 7 checks this by rebuilding `sub-001` and
comparing it with the cached file. It also extends the cache to the
**frontotemporal-dementia** group for the multi-class task:

| subjects | group (`participants.tsv`) | label | windows per subject (cap) |
|---|---|---|---|
| sub-001 … sub-036 | A: Alzheimer's | 1 | first 80 |
| sub-037 … sub-065 | C: healthy control | 0 | first 100 |
| sub-066 … sub-088 | F: frontotemporal dementia | **2** | first 100 |

FTD recordings are shorter (mean ~141 windows, min 92). At a cap of 100, 22 of
the 23 subjects contribute a full cap, so each person carries about the same
weight. The remaining window imbalance (~2,300 FTD vs ~2,900 AD/HC) is left to a
class-weighted loss in the model notebook.

Labels 0/1 are unchanged, so the binary notebooks keep working. They load only
sub-001 … sub-065. The build is **resumable**: existing `.npz` files are skipped.


In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION  (values identical to mfe-fcgcn.ipynb)
# ===========================================================================
import glob, os, time, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_selection import mutual_info_regression
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

WORK_DIR  = Path.cwd()
DATA_ROOT = WORK_DIR / "data"                 # ds004504 v1.0.x, with derivatives/
CACHE_DIR = WORK_DIR / "subject_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PREFER_DERIVATIVES = True       # ASR + ICA cleaned recordings, as the paper uses
FORCE_REBUILD      = False      # True = rebuild even if the .npz exists

# ---- signal / PSD  (paper: "EEG dataset and sample expansion") -------------
FREQ_RES        = 0.2           # Hz
N_FFT           = 2500          # 500 Hz / 0.2 Hz -> 0.2 Hz bins
WELCH_OVERLAP   = 0             # four 5 s segments span exactly 20 s
WINDOW_SEGMENTS = 4             # 4 x 5 s = 20 s window
WINDOW_STEP     = 1             # slide 1 segment = 5 s = 75 % overlap

BAND_NAMES  = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
BAND_RANGES = [(0.5, 4.0), (4.0, 8.0), (8.0, 13.0), (13.0, 30.0), (30.0, 45.0)]
EXPECTED_FEATURE_DIMS = {"Delta": 18, "Theta": 21, "Alpha": 26, "Beta": 86, "Gamma": 76}

CHANNELS = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
            "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
N_NODES = len(CHANNELS)

# ---- classes ---------------------------------------------------------------
GROUP_LABEL = {"C": 0, "A": 1, "F": 2}       # HC, AD, FTD
MAX_SAMPLES = {1: 80, 0: 100, 2: 100}        # first N windows per subject

# ---- features / connectivity (must match the existing cache) ---------------
FEATURE_MODE = "log_global"     # log10 PSD, one mean/std per 20 s window
MI_METHOD    = "knn"            # sklearn Kraskov MI
MI_BINS      = "sqrt"           # only used when MI_METHOD = "hist"

SEED   = 42
N_JOBS = max(1, min(8, (os.cpu_count() or 2) - 2))

print(f"data   : {DATA_ROOT}")
print(f"cache  : {CACHE_DIR}  ({len(list(CACHE_DIR.glob('sub-*.npz')))} files present)")
print(f"workers: {N_JOBS}")


data   : C:\Projects\Major Project\data
cache  : C:\Projects\Major Project\subject_cache  (65 files present)
workers: 8


C:\Projects\Major Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===========================================================================
# CELL 2 — SUBJECT TABLE  (labels read from participants.tsv)
# ===========================================================================
PATTERN = "*_task-eyesclosed_eeg.set"
SET_FILES = sorted(glob.glob(str(DATA_ROOT / "**" / PATTERN), recursive=True))

def is_derivative(path):
    return "derivatives" in Path(path).parts

file_for = {}
for p in SET_FILES:
    sid = Path(p).name.split("_")[0]
    better = sid not in file_for or (PREFER_DERIVATIVES and is_derivative(p)
                                     and not is_derivative(file_for[sid]))
    if better:
        file_for[sid] = p

participants = pd.read_csv(DATA_ROOT / "participants.tsv", sep="\t").set_index("participant_id")
subjects = [(file_for[sid], GROUP_LABEL[participants.loc[sid, "Group"]], sid)
            for sid in participants.index]

# the binary notebooks hard-code these ranges, so they must still hold
for path, label, sid in subjects:
    i = int(sid.split("-")[1])
    assert label == (1 if i <= 36 else 0 if i <= 65 else 2), sid
    assert is_derivative(path), f"{sid}: not using the preprocessed recording"

counts = pd.Series([l for _, l, _ in subjects]).map({0: "HC", 1: "AD", 2: "FTD"}).value_counts()
print(f"subjects: {len(subjects)}  ->  {counts.to_dict()}")


subjects: 88  ->  {'AD': 36, 'HC': 29, 'FTD': 23}


In [3]:
# ============================================================
# CELL 3 — PSD EXTRACTION  (paper Eq. 1)
# ============================================================
# One Welch call for all 19 channels at once (the old notebook made 19 separate
# calls per band, i.e. 95 per subject). average=None keeps the per-segment PSDs
# so we can build 20 s windows by averaging four consecutive 5 s segments.

def extract_band_psd(set_path):
    """Return {band: array (n_segments, 19, n_freq_bins)} of raw PSD."""
    raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose="ERROR")
    sfreq = raw.info["sfreq"]

    missing = [c for c in CHANNELS if c not in raw.info["ch_names"]]
    if missing:
        raise ValueError(f"{Path(set_path).name}: missing channels {missing}")

    expected_nfft = int(round(sfreq / FREQ_RES))
    if expected_nfft != N_FFT:
        raise ValueError(
            f"{Path(set_path).name}: sfreq={sfreq} needs n_fft={expected_nfft} "
            f"for {FREQ_RES} Hz resolution, but N_FFT={N_FFT}"
        )

    data = raw.get_data(picks=CHANNELS)            # (19, T), CHANNELS order
    out = {}

    for band, (fmin, fmax) in zip(BAND_NAMES, BAND_RANGES):
        psds, _ = mne.time_frequency.psd_array_welch(
            data, sfreq, fmin=fmin, fmax=fmax,
            n_fft=N_FFT, n_per_seg=N_FFT, n_overlap=WELCH_OVERLAP,
            average=None, verbose="ERROR",
        )                                          # (19, n_freq, n_seg)

        block = np.transpose(psds, (2, 0, 1)).astype(np.float64)
        got, want = block.shape[2], EXPECTED_FEATURE_DIMS[band]
        if got != want:
            raise ValueError(f"{band}: expected {want} bins, got {got}")
        out[band] = block

    return out

print("PSD extractor defined "
      f"(n_fft={N_FFT}, n_overlap={WELCH_OVERLAP} -> "
      f"{WINDOW_SEGMENTS * N_FFT / 500:.0f} s windows)")

PSD extractor defined (n_fft=2500, n_overlap=0 -> 20 s windows)


In [4]:
# ============================================================
# CELL 4 — FEATURE NORMALISATION  (FIX 4)
# ============================================================
# The paper never describes a normalisation step. The old notebook z-scored
# every channel's PSD vector inside every band, which removes that band's mean
# power and variance -- exactly the quantity that carries the AD signal
# (delta/theta up, alpha/beta down). These modes let you measure that claim.

def normalise_window(block, mode=None):
    """block: {band: (19, F_f)} raw PSD for one 20 s window -> same shape, float32."""
    mode = mode or FEATURE_MODE

    if mode == "none":
        return {b: v.astype(np.float32) for b, v in block.items()}

    if mode == "released":
        out = {}
        for b, v in block.items():
            mu = v.mean(axis=1, keepdims=True)
            sd = np.maximum(v.std(axis=1, keepdims=True), 1e-12)
            out[b] = ((v - mu) / sd).astype(np.float32)
        return out

    logs = {b: np.log10(np.maximum(v, 1e-30)) for b, v in block.items()}

    if mode == "log_perband":
        out = {}
        for b, v in logs.items():
            mu = v.mean(axis=1, keepdims=True)
            sd = np.maximum(v.std(axis=1, keepdims=True), 1e-12)
            out[b] = ((v - mu) / sd).astype(np.float32)
        return out

    # log_global: one centre/scale for the whole window, so the *differences*
    # between bands and between channels survive.
    flat = np.concatenate([v.ravel() for v in logs.values()])
    mu, sd = flat.mean(), max(float(flat.std()), 1e-12)
    return {b: ((v - mu) / sd).astype(np.float32) for b, v in logs.items()}


print(f"normalisation mode: {FEATURE_MODE}")

normalisation mode: log_global


In [5]:
# ============================================================
# CELL 5 — FUNCTIONAL CONNECTIVITY  (paper Eqs. 2-5)
# ============================================================

def pearson_e1(v):
    """Eqs. 2-3. v: (19, F) -> E1 (19, 19).

    Eq. 2 takes |Pearson|; Eq. 3 divides each row by (row sum - 1), i.e. by the
    sum of that row's OFF-diagonal entries, leaving the diagonal at
    1 / off-diagonal-sum rather than at 1.
    """
    r = np.abs(np.corrcoef(v))
    r = np.nan_to_num(r, nan=0.0, posinf=0.0, neginf=0.0)
    denom = np.maximum(r.sum(axis=1) - 1.0, 1e-12)
    return (r / denom[:, None]).astype(np.float32)


def _mi_knn(v):
    """sklearn Kraskov MI. v: (19, F) -> raw MI (19, 19), mi[m, n] = I(X_m ; X_n).

    19 calls instead of 361: mutual_info_regression accepts a 2-D X and scores
    every column against the target in one pass. Identical estimator.
    """
    X = v.T                                        # (F, 19): samples x channels
    out = np.empty((N_NODES, N_NODES), dtype=np.float64)
    for n in range(N_NODES):
        out[:, n] = mutual_info_regression(
            X, X[:, n], discrete_features=False, random_state=SEED)
    return out


def _mi_hist(v, n_bins):
    """Eq. 4 taken literally, vectorised over all 361 pairs at once.

    Equal-frequency (rank) binning, so both marginals are uniform and the
    estimate depends only on how the joint departs from independence.
    """
    F = v.shape[1]
    order = np.argsort(v, axis=1)
    ranks = np.empty_like(order)
    np.put_along_axis(ranks, order, np.arange(F)[None, :].repeat(N_NODES, 0), axis=1)
    codes = (ranks * n_bins // F).astype(np.int64)          # (19, F)

    pair = codes[:, None, :] * n_bins + codes[None, :, :]   # (19, 19, F)
    cell = n_bins * n_bins
    flat = np.arange(N_NODES * N_NODES)[:, None] * cell + pair.reshape(-1, F)

    joint = np.bincount(flat.ravel(), minlength=N_NODES * N_NODES * cell)
    joint = joint.reshape(N_NODES * N_NODES, n_bins, n_bins) / F

    px = joint.sum(axis=2)[:, :, None]
    py = joint.sum(axis=1)[:, None, :]
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = joint * np.log(joint / (px * py))
    terms[~np.isfinite(terms)] = 0.0
    return terms.sum(axis=(1, 2)).reshape(N_NODES, N_NODES)


def mi_e2(v):
    """Eqs. 4-5. v: (19, F) -> E2 (19, 19), each row divided by its maximum."""
    if MI_METHOD == "knn":
        m = _mi_knn(v)
    else:
        n_bins = (int(round(np.sqrt(v.shape[1]))) if MI_BINS == "sqrt"
                  else int(MI_BINS))
        m = _mi_hist(v, max(2, n_bins))
    m = np.maximum(m, 0.0)
    return (m / np.maximum(m.max(axis=1, keepdims=True), 1e-12)).astype(np.float32)


print(f"connectivity ready (MI = {MI_METHOD})")

connectivity ready (MI = knn)


In [6]:
# ============================================================
# CELL 6 — BUILD ONE SUBJECT'S SAMPLES
# ============================================================
# Each sample = one 20 s window = 4 averaged Welch segments, giving for every
# band a (19, F_f) feature matrix plus two 19x19 connectivity matrices.

def build_subject(set_path, label):
    bands = extract_band_psd(set_path)
    n_seg = bands[BAND_NAMES[0]].shape[0]
    cap = MAX_SAMPLES[label]

    xs = {b: [] for b in BAND_NAMES}
    e1s, e2s, bps = [], [], []

    starts = range(0, n_seg - WINDOW_SEGMENTS + 1, WINDOW_STEP)
    for k, s in enumerate(starts):
        if k >= cap:
            break

        raw_block = {b: bands[b][s:s + WINDOW_SEGMENTS].mean(axis=0)
                     for b in BAND_NAMES}                    # (19, F_f)

        # Untouched band power, kept for the Cell 11 sanity baseline only.
        bps.append(np.stack([np.log10(np.maximum(raw_block[b], 1e-30)).mean(axis=1)
                             for b in BAND_NAMES]))          # (5, 19)

        block = normalise_window(raw_block)

        e1_band, e2_band = [], []
        for b in BAND_NAMES:
            v = block[b].astype(np.float64)
            e1_band.append(pearson_e1(v))
            e2_band.append(mi_e2(v))
            xs[b].append(block[b])

        e1s.append(np.stack(e1_band))                        # (5, 19, 19)
        e2s.append(np.stack(e2_band))

    if not e1s:
        raise RuntimeError(f"{Path(set_path).name}: recording too short")

    payload = {f"x_{b}": np.stack(xs[b]).astype(np.float32) for b in BAND_NAMES}
    payload["e1"] = np.stack(e1s).astype(np.float32)
    payload["e2"] = np.stack(e2s).astype(np.float32)
    payload["bp"] = np.stack(bps).astype(np.float32)
    payload["y"] = np.full(len(e1s), label, dtype=np.int64)
    return payload


def process_subject(subject):
    set_path, label, sid = subject
    cache_file = CACHE_DIR / f"{sid}.npz"
    if cache_file.exists() and not FORCE_REBUILD:
        return sid, "cached", 0

    try:
        payload = build_subject(set_path, label)
        # NOTE: np.savez appends ".npz" to any path not already ending in it,
        # so the temp name must itself end in .npz or the rename below
        # looks for a file that was never created.
        tmp = cache_file.with_name(f"{sid}.tmp.npz")
        np.savez(tmp, **payload)
        os.replace(tmp, cache_file)
        return sid, "done", len(payload["y"])
    except Exception as exc:                                  # noqa: BLE001
        return sid, f"error: {exc!r}", 0


print("sample builder ready")

sample builder ready


In [7]:
# ===========================================================================
# CELL 7 — REPRODUCIBILITY CHECK + TIMING
# ===========================================================================
# Rebuild sub-001 in memory and compare it with the cached file. If this passes,
# new subjects are processed exactly like the ones the models were trained on.
# It also times one subject, so you can see how long Cell 8 will take.
_t0 = time.time()
_new = build_subject(*subjects[0][:2])
_secs = time.time() - _t0

with np.load(CACHE_DIR / "sub-001.npz") as _old:
    print(f"sub-001 rebuilt in {_secs:.0f} s  ({len(_new['y'])} windows)")
    _ok = True
    for k in _old.files:
        same_shape = _old[k].shape == _new[k].shape
        diff = float(np.abs(_old[k].astype(np.float64) - _new[k]).max()) if same_shape else np.inf
        _ok &= same_shape and diff < 1e-4
        print(f"  {k:8s} {str(_old[k].shape):18s} max |diff| {diff:.2e}")
print("MATCHES the existing cache" if _ok else "*** DOES NOT MATCH: stop and investigate before building ***")

_todo = [s for s in subjects if FORCE_REBUILD or not (CACHE_DIR / f"{s[2]}.npz").exists()]
print(f"\n{len(_todo)} subjects to build -> about "
      f"{_secs * np.ceil(len(_todo) / N_JOBS) / 60:.0f} min on {N_JOBS} workers")
del _new


sub-001 rebuilt in 100 s  (80 windows)
  x_Delta  (80, 19, 18)       max |diff| 0.00e+00
  x_Theta  (80, 19, 21)       max |diff| 0.00e+00
  x_Alpha  (80, 19, 26)       max |diff| 0.00e+00
  x_Beta   (80, 19, 86)       max |diff| 0.00e+00
  x_Gamma  (80, 19, 76)       max |diff| 0.00e+00
  e1       (80, 5, 19, 19)    max |diff| 0.00e+00
  e2       (80, 5, 19, 19)    max |diff| 0.00e+00
  bp       (80, 5, 19)        max |diff| 0.00e+00
  y        (80,)              max |diff| 0.00e+00
MATCHES the existing cache

23 subjects to build -> about 5 min on 8 workers


In [8]:
# ===========================================================================
# CELL 8 — BUILD MISSING SUBJECTS  (parallel, resumable)
# ===========================================================================
assert _ok, "Cell 7 found a mismatch with the existing cache"
_t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="loky")(
    delayed(process_subject)(s) for s in tqdm(subjects, desc="subjects"))

errors = [(sid, st) for sid, st, _ in results if st.startswith("error")]
print(f"built {sum(st == 'done' for _, st, _ in results)}   "
      f"cached {sum(st == 'cached' for _, st, _ in results)}   errors {len(errors)}   "
      f"[{(time.time() - _t0) / 60:.1f} min]")
for sid, st in errors:
    print(f"  {sid}: {st}")


subjects:   0%|          | 0/88 [00:00<?, ?it/s]

subjects:  18%|█▊        | 16/88 [00:01<00:08,  8.32it/s]

subjects:  36%|███▋      | 32/88 [00:02<00:02, 18.78it/s]

subjects: 100%|██████████| 88/88 [00:02<00:00, 42.23it/s]

built 23   cached 65   errors 0   [8.9 min]


In [9]:
# ===========================================================================
# CELL 9 — WHAT IS IN THE CACHE
# ===========================================================================
rows = []
for _, label, sid in subjects:
    with np.load(CACHE_DIR / f"{sid}.npz") as z:
        assert int(z["y"][0]) == label, f"{sid}: cached label {int(z['y'][0])} != {label}"
        rows.append(dict(sid=sid, cls={0: "HC", 1: "AD", 2: "FTD"}[label], windows=len(z["y"])))
table = pd.DataFrame(rows)
print(table.groupby("cls").windows.agg(subjects="size", windows="sum", min="min", max="max"))
print(f"\ntotal: {len(table)} subjects, {table.windows.sum():,} windows")


     subjects  windows  min  max
cls                             
AD         36     2858   58   80
FTD        23     2292   92  100
HC         29     2900  100  100

total: 88 subjects, 8,050 windows
